In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from DeepScence.api import DeepScence
from SenCID.api import SenCID
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from dca.api import dca
import os
os.chdir(b'/Users/lele/Library/Mobile Documents/com~apple~CloudDocs/Research/Aging')

/Users/lele/Downloads/anaconda3/envs/sene/lib/python3.8/site-packages/kopt/config.py:60: YAMLLoadWarning: calling yaml.load() without Loader=... is deprecated, as the default Loader is unsafe. Please read https://msg.pyyaml.org/load for full details.
  _config = yaml.load(open(_config_path))


### read datasets and gs

In [9]:
all_gs = pd.read_csv("./data/coreGS_v2.csv", index_col=0)
columns_to_use = all_gs.columns[:9]
anchors = {
    "trans": ["STAT1", "Stat1"],
    "network": ["IL6", "Il6"],
    "sensig": [None, None],
    "Senmayo": [None, None],
    "geneAge": [None, None],
    "cellAge": [None, None],
    "CSgene": [None, None],
    "SASP": ["IL6", "Il6"],
    "Quest": [None, None]
}
gs_list_human = {col: all_gs.index[all_gs[col] == True].tolist() for col in columns_to_use}
gs_list_mouse = {col: all_gs["mouse_gene"][all_gs[col] == True].tolist() for col in columns_to_use}

In [ ]:
# # append celltype column and overwrite
# d = "mouse_testes"
# path = f"./data/VADLIATION_DATA/IN_VIVO/Current/h5ad/{d}_dca.h5ad"
# adata = sc.read_h5ad(path)
# adata = adata[adata.obs["subtype"]=="Ld"]
# adata.obs["celltype"] = adata.obs["subtype"]
# print(adata.obs["celltype"].value_counts())
# adata.write_h5ad(path)

### 1. Do existing datasets

In [ ]:
args = {
        "binarize": False,
        "verbose": False
    }
for d in ["human_IPF","mouse_muscle", "mouse_testes", "human_oral"]:
    print(f"processing {d}...")
    adata = sc.read_h5ad(f"./data/VADLIATION_DATA/IN_VIVO/Current/h5ad/{d}_dca.h5ad")
    m = pd.DataFrame(index=adata.obs_names)
    m["celltype"] = adata.obs["celltype"].values

    # DeepScence + all gs, run for each cell type
    for gsname, gs in gs_list_human.items():
        for ct in adata.obs["celltype"].unique():
            this_ct = adata[adata.obs["celltype"]==ct]
            print(f"Running {d} - {ct}: DeepScence + {gsname}")
            anchor = anchors[gsname][0]
            this_ct = DeepScence(this_ct, custome_gs = gs, anchor_gene=anchor, **args)
            m.loc[this_ct.obs_names, f"ds_{gsname}"] = this_ct.obs["ds"].values

    # DeepScence + different n
    for n in [3,4,5,6]:
        for ct in adata.obs["celltype"].unique():
            this_ct = adata[adata.obs["celltype"]==ct]
            print(f"Running {d} - {ct}: DeepScence + >={n}...")
            this_ct = DeepScence(this_ct, n=n, **args)
            m.loc[this_ct.obs_names, f"ds_{n}+"] = this_ct.obs["ds"].values

    # SenCID
    pred_dict, recSID, tmpfiles = SenCID(
                adata=adata,
                sidnums=[1, 2, 3, 4, 5, 6],
                denoising=False,
                binarize=True,
                threads=1,
                savetmp=True,
            )
    binary2 = []
    scores2 = []
    for i in range(len(recSID)):
        rec = recSID["RecSID"].iloc[i]
        score = pred_dict[rec]["SID_Score"].iloc[i]
        b = pred_dict[rec]["Binarization"].iloc[i]
        binary2.append(b)
        scores2.append(score)
    m["SID_binary"] = binary2
    m["SID_score"] = scores2

    # save
    m.to_csv(f"./data/VADLIATION_DATA/IN_VIVO/Current/metas/{d}_scores_part2.csv")

### 2. Do new datasets

In [ ]:
args = {
        "binarize": False,
        "verbose": False
    }
for d in ["mouse_lung_cancer","mouse_tauopathy", "mouse_heart_CMs", "mouse_heart_ECs"]:
    adata = sc.read_h5ad(f"./data/VADLIATION_DATA/IN_VIVO/Current/h5ad/{d}_dca.h5ad")
    m = pd.DataFrame(index=adata.obs_names)
    m["celltype"] = adata.obs["celltype"].values

    # DeepScence + all gs, run for each cell type
    for gsname, gs in gs_list_mouse.items():
        for ct in adata.obs["celltype"].unique():
            this_ct = adata[adata.obs["celltype"]==ct]
            anchor = anchors[gsname][1]
            this_ct = DeepScence(this_ct, custome_gs = gs, anchor_gene=anchor, **args)
            m.loc[this_ct.obs_names, f"ds_{gsname}"] = this_ct.obs["ds"].values

    # DeepScence + different n
    for n in [3,4,5,6]:
        for ct in adata.obs["celltype"].unique():
            this_ct = adata[adata.obs["celltype"]==ct]
            print(f"Running {d} - {ct}: DeepScence + >={n}...")
            this_ct = DeepScence(this_ct, n=n, species="mouse", **args)
            m.loc[this_ct.obs_names, f"ds_{n}+"] = this_ct.obs["ds"].values

    # SenCID

    # to run SenCID, we need to substitute sencid genes into human homologs
    adata.var['gene_symbols'] = adata.var.index
    c = pd.read_csv("./data/in_vivo/gene_convert.csv")
    gene_map = dict(zip(c['original'].dropna(), c['converted'].dropna()))
    adata.var["converted_gene"] = adata.var["gene_symbols"].map(lambda x: gene_map.get(x, x))
    adata.var_names = adata.var["converted_gene"].values
    adata.var_names_make_unique()
    
    pred_dict, recSID, tmpfiles = SenCID(
                adata=adata,
                sidnums=[1, 2, 3, 4, 5, 6],
                denoising=False,
                binarize=True,
                threads=1,
                savetmp=True,
            )
    binary2 = []
    scores2 = []
    for i in range(len(recSID)):
        rec = recSID["RecSID"].iloc[i]
        score = pred_dict[rec]["SID_Score"].iloc[i]
        b = pred_dict[rec]["Binarization"].iloc[i]
        binary2.append(b)
        scores2.append(score)
    m["SID_binary"] = binary2
    m["SID_score"] = scores2
    
    # save
    m.to_csv(f"./data/VADLIATION_DATA/IN_VIVO/Current/metas/{d}_scores_part2.csv")

### 3. Do standard

In [ ]:
args = {
        "binarize": False,
        "verbose": False,
        "random_state": 11
    }
for d in ["human_IPF", "human_oral", "mouse_muscle", "mouse_testes", "mouse_lung_cancer","mouse_tauopathy", "mouse_heart_CMs", "mouse_heart_ECs"]:
    adata = sc.read_h5ad(f"./data/VADLIATION_DATA/IN_VIVO/Standard/h5ad/{d}_dca.h5ad")
    m = pd.DataFrame(index=adata.obs_names)
    m["celltype"] = adata.obs["celltype"].values

    if d.startswith("mouse_"):
        gs_list = gs_list_mouse
        species = "mouse"
        use = 1
    else:
        gs_list = gs_list_human
        species = "human"
        use = 0

    # DeepScence + different n
    for n in [3,4,5,6]:
        for ct in adata.obs["celltype"].unique():
            this_ct = adata[adata.obs["celltype"]==ct]
            print(f"Running {d} - {ct}: DeepScence + >={n}...")
            this_ct = DeepScence(this_ct, n=n, species=species, **args)
            print(this_ct.uns["log"]["corr_df"])
            m.loc[this_ct.obs_names, f"ds_{n}+"] = this_ct.obs["ds"].values
        
    # DeepScence + all gs, run for each cell type
    for gsname, gs in gs_list.items():
        for ct in adata.obs["celltype"].unique():
            this_ct = adata[adata.obs["celltype"]==ct]
            print(f"Running {d} - {ct}: DeepScence + {gsname}")
            this_ct = DeepScence(this_ct, custome_gs = gs, anchor_gene=anchors[gsname][use], **args)
            m.loc[this_ct.obs_names, f"ds_{gsname}"] = this_ct.obs["ds"].values


    # SenCID

    # to run SenCID, we need to substitute sencid genes into human homologs
    if d.startswith("mouse_"):
        adata.var['gene_symbols'] = adata.var.index
        c = pd.read_csv("./data/in_vivo/gene_convert.csv")
        gene_map = dict(zip(c['original'].dropna(), c['converted'].dropna()))
        adata.var["converted_gene"] = adata.var["gene_symbols"].map(lambda x: gene_map.get(x, x))
        adata.var_names = adata.var["converted_gene"].values
        adata.var_names_make_unique()
    
    pred_dict, recSID, tmpfiles = SenCID(
                adata=adata,
                sidnums=[1, 2, 3, 4, 5, 6],
                denoising=False,
                binarize=True,
                threads=1,
                savetmp=True,
            )
    binary2 = []
    scores2 = []
    for i in range(len(recSID)):
        rec = recSID["RecSID"].iloc[i]
        score = pred_dict[rec]["SID_Score"].iloc[i]
        b = pred_dict[rec]["Binarization"].iloc[i]
        binary2.append(b)
        scores2.append(score)
    m["SID_binary"] = binary2
    m["SID_score"] = scores2
    
    # save
    m.to_csv(f"./data/VADLIATION_DATA/IN_VIVO/Standard/metas/{d}_scores_part2.csv")

### Driving gene analysis

In [5]:
gs

,trans,network,sensig,Senmayo,geneAge,cellAge,CSgene,SASP,Quest,n,mouse_gene
IL6,False,True,False,True,True,True,True,True,True,7,Il6
IGFBP3,False,True,True,True,True,True,True,False,True,7,Igfbp3
EGFR,False,True,False,True,True,True,True,True,True,7,Egfr
SERPINE1,False,True,False,True,True,True,False,True,True,6,Serpine1
IGFBP1,False,True,False,True,False,True,True,True,True,6,Igfbp1
...,...,...,...,...,...,...,...,...,...,...,...
STAT31,False,False,False,False,False,False,False,False,True,1,NaN
TERT1,False,False,False,False,False,False,False,False,True,1,NaN
TGFB11,False,False,False,False,False,False,False,False,True,1,NaN
TNF1,False,False,False,False,False,False,False,False,True,1,NaN


In [6]:
args = {
    "binarize": False,
    "verbose": False
}

gs = pd.read_csv("./data/coreGS_v2.csv", index_col=0)
core_human = gs.index[gs["n"] >= 5].tolist()
core_mouse = gs["mouse_gene"][gs["n"]>=5].tolist()
mouse_to_human = dict(zip(core_mouse, core_human))
res_list = []

# Loop over datasets
for d in ["mouse_testes", "human_IPF", "human_oral", "mouse_muscle", "mouse_lung_cancer",
          "mouse_tauopathy", "mouse_heart_CMs", "mouse_heart_ECs"]:
    adata = sc.read_h5ad(f"./data/VADLIATION_DATA/IN_VIVO/Current/h5ad/{d}_dca.h5ad")

    # adhoc fix for species
    if d in ["mouse_lung_cancer","mouse_tauopathy", "mouse_heart_CMs", "mouse_heart_ECs"]:
        species = "mouse"
    else:
        species = "human"
    
    for ct in adata.obs["celltype"].unique():
        this_ct = adata[adata.obs["celltype"] == ct].copy()
        this_ct = DeepScence(this_ct, species=species, **args)
        corr_df = this_ct.uns["log"]["corr_df"]
        

        if species == "mouse":
            corr_df["gene_symbol"] = corr_df["gene_symbol"].apply(lambda x: mouse_to_human.get(x, x))
        print(corr_df)  # for debugging
        
        row_name = f"{d}_{ct}"
        gene_corr = dict(zip(corr_df["gene_symbol"], corr_df["correlation"]))
        row_data = {gene: gene_corr.get(gene, np.nan) for gene in core}
        row_data["dataset_celltype"] = row_name
        
        # Append the row dictionary to the list
        res_list.append(row_data)

res = pd.DataFrame(res_list)
res.set_index("dataset_celltype", inplace=True)
print(res)

[2025-03-28 00:33] GPU not available, using CPU...
[2025-03-28 00:33] Input is not count, processed 26583 genes and 41 cells.
[2025-03-28 00:33] Using 35 genes in the gene set for scoring.
[2025-03-28 00:33] Lambda provided, capturing scores in 2 neurons.
[2025-03-28 00:33] Training on 37 cells, validate on 4 cells.
100%|████████████████████████████████████████| 300/300 [00:00<00:00, 308.57it/s]
[2025-03-28 00:33] GPU not available, using CPU...
[2025-03-28 00:33] Input is not count, processed 11423 genes and 135 cells.
[2025-03-28 00:33] Using 29 genes in the gene set for scoring.
[2025-03-28 00:33] Lambda provided, capturing scores in 2 neurons.
[2025-03-28 00:33] Training on 122 cells, validate on 13 cells.


   gene_symbol  correlation
0       IGFBP5     0.964640
1        LMNB1     0.961801
2         IL1A     0.943666
3       CDKN2A     0.936770
4       IGFBP3     0.935524
5         FGF2     0.934328
6         CDK1     0.913050
7        TGFB1     0.894453
8     SERPINE1     0.887926
9         WNT2     0.883571
10       PARP1     0.875300
11      CDKN2B     0.854566
12       BUB1B     0.831310
13      IGFBP1     0.825818
14        CCL2     0.808713
15         AXL     0.808356
16       CCNA2     0.794211
17       ICAM1     0.764690
18       STAT1     0.740545
19       GDF15     0.726188
20       VEGFA     0.687472
21        MDM2     0.663329
22       FOXM1     0.639408
23       HELLS     0.566728
24       BRCA1     0.497346
25       HMGB2     0.459301
26        IGF1     0.416506
27      CDKN1A     0.378989
28         FAS     0.233859
29        EGFR     0.196845
30       HMGB1    -0.080251
31      IGFBP2    -0.095740
32         MIF    -0.096185
33         JUN    -0.227123
34      IGFBP7    -0

 37%|██████████████▉                         | 112/300 [00:00<00:00, 205.71it/s]


   gene_symbol  correlation
0        CXCL1     0.958657
1        TGFB1     0.954870
2          FAS     0.948705
3       CDKN1A     0.944267
4        CXCL8     0.936605
5       CDKN2B     0.902206
6        GDF15     0.900644
7          AXL     0.894688
8       CDKN2A     0.884594
9       IGFBP3     0.856357
10        CCL2     0.853655
11       ICAM1     0.718193
12      IGFBP2     0.661211
13      IGFBP7     0.644961
14        MDM2     0.608677
15       PARP1     0.576344
16         JUN     0.532222
17         MIF     0.183279
18       VEGFA     0.112889
19       HMGB2     0.107192
20       LMNB1    -0.206215
21         IL6    -0.210956
22      IGFBP5    -0.226718
23       STAT1    -0.408153
24       BRCA1    -0.503070
25        CDK1    -0.510684
26        EGFR    -0.588775
27       HELLS    -0.699854
28       HMGB1    -0.703429


[2025-03-28 00:34] GPU not available, using CPU...
[2025-03-28 00:34] Input is not count, processed 12232 genes and 14519 cells.
[2025-03-28 00:34] Using 31 genes in the gene set for scoring.
[2025-03-28 00:34] Lambda provided, capturing scores in 2 neurons.
[2025-03-28 00:34] Training on 13068 cells, validate on 1451 cells.
100%|█████████████████████████████████████████| 300/300 [01:44<00:00,  2.86it/s]
[2025-03-28 00:36] GPU not available, using CPU...


...

   gene_symbol  correlation
0          AXL     0.793580
1          MIF     0.770019
2         CDK1     0.713737
3         IGF1     0.677180
4         CCL2     0.675343
5       CDKN2A     0.595419
6          FAS     0.509455
7        LMNB1     0.401245
8       CDKN1A     0.331981
9         FGF2     0.289032
10       STAT1     0.187157
11       HMGB1     0.033655
12       HELLS     0.001698
13      IGFBP5    -0.008497
14      IGFBP2    -0.034203
15       VEGFA    -0.069885
16       BUB1B    -0.154087
17       HMGB2    -0.231927
18    SERPINE1    -0.282485
19       PARP1    -0.354620
20        IL1A    -0.438309
21       ICAM1    -0.459673
22   TNFRSF10C    -0.665303
23        MDM2    -0.695435
24       TGFB1    -0.730736
25         IL6    -0.789639
26         JUN    -0.898027


[2025-03-28 00:38] Input is not count, processed 32901 genes and 481 cells.
[2025-03-28 00:38] Using 35 genes in the gene set for scoring.
[2025-03-28 00:38] Lambda provided, capturing scores in 2 neurons.
[2025-03-28 00:38] Training on 433 cells, validate on 48 cells.
100%|█████████████████████████████████████████| 300/300 [00:03<00:00, 77.98it/s]
[2025-03-28 00:38] GPU not available, using CPU...


   gene_symbol  correlation
0         IGF1     0.872910
1        HMGB2     0.866015
2       CDKN2B     0.864689
3        CXCL1     0.854527
4        ICAM1     0.846588
5         IL1A     0.845019
6         WNT2     0.823075
7          IL6     0.815084
8        FOXM1     0.806243
9        BUB1B     0.763951
10      IGFBP3     0.759190
11      IGFBP5     0.749096
12      IGFBP2     0.746553
13       CCNA2     0.746399
14        CDK1     0.729716
15       TGFB1     0.728100
16       BRCA1     0.685297
17        EGFR     0.671804
18         FAS     0.657950
19      CDKN1A     0.656557
20       HELLS     0.653851
21       STAT1     0.625221
22      IGFBP7     0.624263
23         JUN     0.613246
24       LMNB1     0.605609
25    SERPINE1     0.596847
26       GDF15     0.592911
27         MIF     0.560753
28       PARP1     0.542751
29   TNFRSF10C     0.540756
30       HMGB1     0.522294
31        FGF2     0.491879
32        MDM2     0.462189
33       VEGFA     0.458546
34         AXL     0

[2025-03-28 00:38] Input is not count, processed 13572 genes and 561 cells.
[2025-03-28 00:38] Using 31 genes in the gene set for scoring.
[2025-03-28 00:38] Lambda provided, capturing scores in 2 neurons.
[2025-03-28 00:38] Training on 505 cells, validate on 56 cells.
100%|█████████████████████████████████████████| 300/300 [00:03<00:00, 76.39it/s]

   gene_symbol  correlation
0       CDKN1A     0.941512
1          FAS     0.815123
2        HMGB1     0.536257
3        TGFB1     0.491603
4       IGFBP3     0.491562
5       IGFBP7     0.446394
6          JUN     0.446028
7        ICAM1     0.375769
8    TNFRSF10C     0.337161
9         FGF2     0.237830
10         IL6     0.218742
11        MDM2     0.209622
12       BRCA1     0.207805
13       BUB1B     0.169684
14       FOXM1     0.084426
15       STAT1    -0.001240
16       CCNA2    -0.002096
17        CDK1    -0.017860
18       HMGB2    -0.034367
19       HELLS    -0.039187
20       PARP1    -0.050827
21       GDF15    -0.083988
22         MIF    -0.087092
23      CDKN2B    -0.165609
24        IGF1    -0.327817
25        EGFR    -0.435614
26       LMNB1    -0.453882
27      IGFBP5    -0.466519
28         AXL    -0.479778
29       VEGFA    -0.646445
30    SERPINE1    -0.694709
                                            IL6    IGFBP3      EGFR  SERPINE1  \
dataset_celltype       

In [8]:
res.to_csv("./data/driving_genes.csv", index=True)

# 